# CTCF + NT attention-guided motif discovery demo

This notebook demonstrates the discovery workflow from precomputed NT attention parquet files to TF-MoDISco inputs and downstream summaries. The historical folder uses the word `recovery`; in the Exp2 codebase this workflow is named `discovery` because it discovers motifs without FIMO coordinates.

In [1]:
from pathlib import Path
import os
import sys

EXP2_ROOT = Path("/dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis")
SRC_ROOT = EXP2_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

LEGACY_MOTIF_ROOT = Path("/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif")
PEAKS_DIR = LEGACY_MOTIF_ROOT / "Data" / "original_peaks"
REFERENCE_DIR = PEAKS_DIR / "reference"
TF_NAME = "CTCF"
MODEL_TYPE = "NT"

print(f"EXP2_ROOT = {EXP2_ROOT}")
print(f"PEAKS_DIR = {PEAKS_DIR}")
print(f"REFERENCE_DIR = {REFERENCE_DIR}")

ATTENTION_DIR = Path("/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_recovery/DNABERT_recovery/predict_true/NT/attention/CTCF_attention_weight")
TRAIN_TRUE_CSV = Path("/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_recovery/DNABERT_recovery/predict_true/NT/CTCF_train_true.csv")
NT_CHECKPOINT = Path("/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/finetune/outputs/NT/motif_CTCF/checkpoint-10720")
print(f"ATTENTION_DIR = {ATTENTION_DIR}")
print("attention dir exists:", ATTENTION_DIR.exists())
print("train_true csv exists:", TRAIN_TRUE_CSV.exists())

EXP2_ROOT = /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis
PEAKS_DIR = /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks
REFERENCE_DIR = /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks/reference
ATTENTION_DIR = /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_recovery/DNABERT_recovery/predict_true/NT/attention/CTCF_attention_weight
attention dir exists: True
train_true csv exists: True


In [2]:
import pandas as pd

part_files = sorted(ATTENTION_DIR.glob("part_*.parquet"))
print(f"attention parquet parts: {len(part_files)}")
if part_files:
    preview_cols = ["sequence_name", "layer", "head", "attention_vector"]
    existing_cols = pd.read_parquet(part_files[0]).columns.tolist()
    cols = [c for c in preview_cols if c in existing_cols]
    display(pd.read_parquet(part_files[0], columns=cols).head())

attention parquet parts: 16


,sequence_name,layer,head,attention_vector
0,seq_0,0,0,"[0.0033706428948789835, 0.0037564977537840605,..."
1,seq_0,0,1,"[0.006019369699060917, 0.004050565883517265, 0..."
2,seq_0,0,2,"[0.007645105943083763, 0.004339697305113077, 0..."
3,seq_0,0,3,"[0.005844352766871452, 0.004355068784207106, 0..."
4,seq_0,0,4,"[0.0058804042637348175, 0.005663941148668528, ..."


In [3]:
# Step 1 can be skipped here because TRAIN_TRUE_CSV and ATTENTION_DIR already exist.
predict_true_command = f"""
python -m exp2_attention.discovery.predict_true \\
  --tf_name {TF_NAME} \\
  --model_type NT \\
  --model_path {NT_CHECKPOINT} \\
  --output_path {TRAIN_TRUE_CSV}

python -m exp2_attention.attention.extract_nt_attention \\
  --tf_name {TF_NAME} \\
  --model_path {NT_CHECKPOINT} \\
  --input_path {TRAIN_TRUE_CSV} \\
  --output_path {ATTENTION_DIR.with_suffix('.parquet')}
"""
print(predict_true_command)


python -m exp2_attention.discovery.predict_true \
  --tf_name CTCF \
  --model_type NT \
  --model_path /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/finetune/outputs/NT/motif_CTCF/checkpoint-10720 \
  --output_path /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_recovery/DNABERT_recovery/predict_true/NT/CTCF_train_true.csv

python -m exp2_attention.attention.extract_nt_attention \
  --tf_name CTCF \
  --model_path /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/finetune/outputs/NT/motif_CTCF/checkpoint-10720 \
  --input_path /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_recovery/DNABERT_recovery/predict_true/NT/CTCF_train_true.csv \
  --output_path /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_r

In [4]:
# Step 2: convert NT token-level attention fold-change to base-level TF-MoDISco inputs.
OUT_DIR = EXP2_ROOT / "outputs" / "motif_discovery" / "predict_true" / "NT" / "tfmodisco_inputs" / TF_NAME
build_inputs_command = f"""
python -m exp2_attention.discovery.expand_nt_attention_to_tfmodisco \\
  --tf_name {TF_NAME} \\
  --csv_path {TRAIN_TRUE_CSV} \\
  --attn_dir {ATTENTION_DIR} \\
  --out_dir {OUT_DIR} \\
  --center none \\
  --fake_negative \\
  --fake_neg_base A \\
  --fake_neg_scale 0.01
"""
print(build_inputs_command)


python -m exp2_attention.discovery.expand_nt_attention_to_tfmodisco \
  --tf_name CTCF \
  --csv_path /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_recovery/DNABERT_recovery/predict_true/NT/CTCF_train_true.csv \
  --attn_dir /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/motif_recovery/DNABERT_recovery/predict_true/NT/attention/CTCF_attention_weight \
  --out_dir /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis/outputs/motif_discovery/predict_true/NT/tfmodisco_inputs/CTCF \
  --center none \
  --fake_negative \
  --fake_neg_base A \
  --fake_neg_scale 0.01



In [5]:
# Step 3: run TF-MoDISco and summarize discovered patterns.
modisco_h5 = OUT_DIR / "modisco_results.h5"
summary_csv = OUT_DIR / "patterns_summary.csv"
jaspar_meme = EXP2_ROOT / "Data" / "meme" / "JASPAR2024_CORE_non-redundant_pfms_meme.txt"

modisco_command = f"""
modisco motifs -s {OUT_DIR / 'ohe1.npz'} -a {OUT_DIR / 'hypscores1.npz'} \\
  -o {modisco_h5} -n 5000 -w 200 -v

modisco meme -i {modisco_h5} -t CWM-PFM -o {OUT_DIR / 'modisco_results.CWM-PFM.meme'} -q

python -m exp2_attention.discovery.summarize_modisco_h5 \\
  -i {modisco_h5} \\
  -o {summary_csv} \\
  -m {jaspar_meme} \\
  --tf {TF_NAME} \\
  --model NT \\
  --query_matrix CWM_PFM
"""
print(modisco_command)


modisco motifs -s /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis/outputs/motif_discovery/predict_true/NT/tfmodisco_inputs/CTCF/ohe1.npz -a /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis/outputs/motif_discovery/predict_true/NT/tfmodisco_inputs/CTCF/hypscores1.npz \
  -o /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis/outputs/motif_discovery/predict_true/NT/tfmodisco_inputs/CTCF/modisco_results.h5 -n 5000 -w 200 -v

modisco meme -i /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis/outputs/motif_discovery/predict_true/NT/tfmodisco_inputs/CTCF/modisco_results.h5 -t CWM-PFM -o /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis/outputs/motif_discovery/predict_true/NT/tfmodisco_inputs/CTCF/modisco_results.CWM-PFM.meme -q

python -m exp2_attention.discovery.summarize_modisco_h5 \
  -i /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis/outputs/motif_discover